# 04 — Full Pipeline Test

End-to-end pipeline: data → features → inference → postprocess → ensemble → submission.
Validates the entire workflow on the validation set before Kaggle submission.

In [ ]:
# === Colab Setup Cell ===
!pip install kaggle -q
import os
from google.colab import files
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file")
    uploaded = files.upload()
    !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

REPO_URL = "https://github.com/YOUR_USER/3drna_cc.git"  # <-- UPDATE THIS
if not os.path.exists('/content/3drna_cc'):
    !git clone {REPO_URL} /content/3drna_cc
%cd /content/3drna_cc

from src.setup import setup_environment
setup_environment()

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

from src.config import OUTPUT_DIR, LORA_WEIGHTS
from src.data.loader import load_sequences, load_labels, labels_to_coords
from src.data.featurizer import build_protenix_input, save_input_json
from src.template.template_search import search_templates
from src.model.protenix_runner import ProtenixRunner
from src.postprocess.clash_fix import fix_all
from src.postprocess.geometry_check import validate_geometry
from src.ensemble.diversity_selector import select_best_five
from src.ensemble.tm_score import best_of_n_tm_score
from src.submission.formatter import format_submission, validate_submission

## Full Pipeline on Validation Set

In [ ]:
val_seq = load_sequences(split='val')
val_labels = load_labels(split='val')

runner = ProtenixRunner(lora_dir=LORA_WEIGHTS, device='cuda')

all_predictions = {}  # target_id -> list of 5 (L,3) arrays
tm_results = []

total_start = time.time()

for idx, row in val_seq.iterrows():
    tid = row['target_id']
    print(f"\n{'='*60}\nTarget: {tid} ({idx+1}/{len(val_seq)})")
    t0 = time.time()
    
    # 1. Template search
    templates = search_templates(tid, row['sequence'])
    print(f"  Templates found: {len(templates)}")
    
    # 2. Build input (with multiple seeds for diversity)
    seeds_list = [list(range(1, 6)), list(range(6, 11))]  # 10 total
    all_preds = []
    
    for seeds in seeds_list:
        inp = build_protenix_input(
            target_id=tid,
            sequence=row['sequence'],
            stoichiometry=row.get('stoichiometry', ''),
            all_sequences=row.get('all_sequences', ''),
            ligand_ids=row.get('ligand_ids', ''),
            ligand_smiles=row.get('ligand_smiles', ''),
            template_hits=templates,
            model_seeds=seeds,
        )
        
        # 3. Run inference
        try:
            cifs = runner.predict_from_dict(inp)
            for cif in cifs:
                coords = runner.extract_c1_prime(cif)
                all_preds.append({'coords': coords, 'source': f'seeds_{seeds}'})
        except Exception as e:
            print(f"  WARNING: inference failed for seeds {seeds}: {e}")
    
    # 4. Post-process
    for p in all_preds:
        p['coords'] = fix_all(p['coords'])
    
    # 5. Select best 5
    if all_preds:
        selected = select_best_five(all_preds, strategy='maxmin_diversity', n_select=5)
    else:
        selected = [np.zeros((len(row['sequence']), 3))] * 5
    
    all_predictions[tid] = selected
    
    # 6. Evaluate
    ref = labels_to_coords(val_labels, tid)
    if len(ref) > 0 and all_preds:
        best_tm, best_idx = best_of_n_tm_score(selected, ref)
        tm_results.append({'target_id': tid, 'tm': best_tm})
        print(f"  TM-score: {best_tm:.4f} (best model {best_idx})")
    
    print(f"  Time: {time.time()-t0:.1f}s")

total_time = time.time() - total_start
print(f"\n{'='*60}")
print(f"Total time: {total_time/60:.1f} min")

tm_df = pd.DataFrame(tm_results)
print(f"Mean TM-score: {tm_df['tm'].mean():.4f}")
print(f"Median: {tm_df['tm'].median():.4f}")

## Generate & Validate Submission

In [ ]:
# Format submission (using validation set as a test run)
submission = format_submission(
    all_predictions,
    output_path=OUTPUT_DIR / 'test_submission.csv'
)

# Validate format
is_valid = validate_submission(submission)
submission.head()